In [ ]:
# Step 1 - Import Libraries
# tensorflow is our main deep learning framework
import tensorflow as tf

# matplotlib lets us visualize images and training results
import matplotlib.pyplot as plt

# numpy helps us work with numbers and arrays
import numpy as np

# partial lets us create a "default" version of a function with preset settings
# we'll use this later to avoid repeating ourselves when building the CNN
from functools import partial

In [ ]:
# Step 2 - Load the Data

# load training images from the train folder
# image_size resizes all images to 64x64 pixels so they're all the same size
# batch_size=32 means we feed 32 images at a time to the model during training
train_ds = tf.keras.utils.image_dataset_from_directory(
    r"C:/Users/KOKO/Desktop/FruitClassifier/dataset/train",
    image_size=(224, 224),
    batch_size=32
)

# load test images — same settings, different folder
test_ds = tf.keras.utils.image_dataset_from_directory(
    r"C:/Users/KOKO/Desktop/FruitClassifier/dataset/test",
    image_size=(224, 224),
    batch_size=32
)

In [ ]:
# Step 2.5 - Visualize Some Training Images
# this helps us make sure the data loaded correctly before we do anything else

# create a 10x10 figure to display our images
plt.figure(figsize=(10, 10))

# take(1) grabs just the first batch of 32 images
for images, labels in train_ds.take(1):
    
    # show 9 images in a 3x3 grid
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        
        # imshow expects values 0-255, so we convert back with astype("uint8")
        plt.imshow(images[i].numpy().astype("uint8"))
        
        # class_names gives us the fruit name from the label number e.g. 0 → "Apple"
        plt.title(train_ds.class_names[labels[i]])
        plt.axis("off")

plt.show()

In [ ]:
# Step 3 - Normalize the Data

# pixel values in images range from 0 to 255 (e.g. 128, 200, 45...)
# neural networks work much better when numbers are small (between 0 and 1)
# so we divide every pixel by 255 to shrink the range → 0.0 to 1.0
normalization_layer = tf.keras.layers.Rescaling(1./255)

# apply the normalization to every image in the training set
# lambda x, y means: for each (image, label) pair, rescale the image, keep the label as is
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))

# same for the test set
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
# Step 4 - Data Augmentation
# remember our overfitting problem? training 64% but validation only 46%?
# augmentation helps fix this by creating new variations of our training images
# so the model sees more diversity and can't just memorize the training set

data_augmentation = tf.keras.Sequential([
    # randomly flip the image horizontally (mirror effect)
    # an apple is still an apple whether flipped or not!
    tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
    
    # randomly rotate the image slightly (max 5% of 360° = ~18 degrees)
    # a mango is still a mango even if slightly tilted
    tf.keras.layers.RandomRotation(factor=0.05, seed=42),
    
    # randomly adjust the contrast slightly
    # helps the model handle different lighting conditions
    tf.keras.layers.RandomContrast(factor=0.2, seed=42)
])

In [ ]:
# Step 5 - Load Pretrained MobileNetV2 (Transfer Learning)

# weights= local path because we downloaded the file manually
# include_top=False means we remove its last layer (which was for 1000 ImageNet classes)
# we'll add our own last layer for our 10 fruits instead
# input_shape=[224, 224, 3] tells it our image size (MobileNetV2 was trained on 224x224)
base_model = tf.keras.applications.MobileNetV2(
    weights=r"C:\Users\KOKO\Favorites\Downloads\mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5",
    include_top=False,
    input_shape=[224, 224, 3]  # ← fixed!
)

# freeze the base model — we don't want to change its pretrained weights yet
# it already knows how to detect shapes, edges, textures
# we just want to use that knowledge, not destroy it!
base_model.trainable = False

print("Base model loaded and frozen ✅")
print(f"Total layers: {len(base_model.layers)}")

In [ ]:
# Step 6 - Build the Transfer Learning Model

model = tf.keras.Sequential([
    # augmentation layer first
    data_augmentation,
    
    # the pretrained MobileNetV2 base — all 154 layers of knowledge!
    base_model,
    
    # GlobalAveragePooling squashes the output of MobileNetV2
    # from (7, 7, 1280) → (1280,) — a flat vector of 1280 features
    tf.keras.layers.GlobalAveragePooling2D(),
    
    # Dropout to prevent overfitting
    tf.keras.layers.Dropout(0.2),
    
    # our final layer — 10 neurons for 10 fruits
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Step 7 - Train the Model
history = model.fit(train_ds, epochs=20, validation_data=test_ds)